# Code 8.5.2: Greedy, beam search, and sampling with GPT-2

Generates 60-token continuations of the city-council prompt from GPT-2 small with a key-value cache, checks the greedy loop against Hugging Face's `generate` and the top-p filter against Hugging Face's `TopPLogitsWarper`, and measures repetition and diversity for each decoding setting.


In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from transformers.generation.logits_process import TopPLogitsWarper
from sampling import sample_next                       # Code 8.5.1

tok = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
prompt = "The city council met on Tuesday night to discuss the plan for the new park."
ids = tok(prompt, return_tensors="pt").input_ids

@torch.no_grad()
def generate(ids, max_new=60, seed=0, **kw):
    """The autoregressive loop with a key-value cache: the prompt once, then one token per step."""
    g = torch.Generator().manual_seed(seed)
    out = model(ids, use_cache=True)
    past, logits, new = out.past_key_values, out.logits[0, -1], []
    for _ in range(max_new):
        t = sample_next(logits.clone(), generator=g, **kw)
        if t == tok.eos_token_id:
            break
        new.append(t)
        out = model(torch.tensor([[t]]), past_key_values=past, use_cache=True)
        past, logits = out.past_key_values, out.logits[0, -1]
    return new

def repeated_4grams(t):                  # fraction of 4-grams that already occurred earlier in the text
    grams = [tuple(t[i:i + 4]) for i in range(len(t) - 3)]
    return sum(g in set(grams[:i]) for i, g in enumerate(grams)) / len(grams)

def distinct_2(samples):                 # distinct bigrams / total bigrams, pooled over samples
    grams = [tuple(s[i:i + 2]) for s in samples for i in range(len(s) - 1)]
    return len(set(grams)) / len(grams)

# Our greedy loop matches Hugging Face's greedy generate
greedy = generate(ids, temperature=0)
hf_greedy = model.generate(ids, max_new_tokens=60, do_sample=False, pad_token_id=tok.eos_token_id)
print(greedy == hf_greedy[0, ids.size(1):].tolist())
print(repr(tok.decode(greedy)))
beam = model.generate(ids, max_new_tokens=60, num_beams=5, do_sample=False,
                      pad_token_id=tok.eos_token_id)[0, ids.size(1):].tolist()
print(repr(tok.decode(beam)))

settings = {"pure sampling": dict(), "temperature 0.7": dict(temperature=0.7),
            "top-k 40": dict(top_k=40), "top-p 0.95": dict(top_p=0.95)}
print(f"{'greedy':16s} repeated 4-grams {repeated_4grams(greedy):.2f}")
print(f"{'beam 5':16s} repeated 4-grams {repeated_4grams(beam):.2f}")
for name, kw in settings.items():
    samples = [generate(ids, seed=s, **kw) for s in range(10)]
    rep = sum(repeated_4grams(s) for s in samples) / len(samples)
    print(f"{name:16s} repeated 4-grams {rep:.2f}, distinct-2 {distinct_2(samples):.2f}")
for s in range(2):
    print(repr(tok.decode(generate(ids, seed=s, top_p=0.95))))
print(repr(tok.decode(generate(ids, seed=0))))                   # pure sampling

# Our top-p filter keeps the same tokens as Hugging Face's TopPLogitsWarper
logits = model(ids).logits[0, -1].detach()
warped = TopPLogitsWarper(top_p=0.9)(ids, logits[None].clone())[0]
probs, order = logits.softmax(-1).sort(descending=True)
ours = order[: int(((probs.cumsum(-1) - probs) < 0.9).sum())]
print(torch.equal(ours.sort().values, torch.isfinite(warped).nonzero().flatten()))
# True
# '\n\nThe city council voted to approve the plan for the new park on Tuesday night. (CBC)\n\nThe city council voted to approve the plan for the new park on Tuesday night.\n\nThe city council voted to approve the plan for the new park on Tuesday night.\n\nThe'
# '\n\n"I think it\'s a good idea, and I think it\'s a good idea for the future of the park," Councillor Doug Ford said.\n\n"I think it\'s a good idea, and I think it\'s a good idea for the future of the park."\n'
# greedy           repeated 4-grams 0.58
# beam 5           repeated 4-grams 0.44
# pure sampling    repeated 4-grams 0.00, distinct-2 0.91
# temperature 0.7  repeated 4-grams 0.00, distinct-2 0.81
# top-k 40         repeated 4-grams 0.00, distinct-2 0.87
# top-p 0.95       repeated 4-grams 0.00, distinct-2 0.88
# ' The existing seven-story building has been "tree-lined" on both sides, and new condos would include retail and restaurants.\n\nThe change would lower parking in a large downtown area which currently pays for gas and a key component of the new park.\n\nKitchener said the city'
# "\n\nAs the news broke, the controversy over the lack of funding for the park soared into the air from the park's website, where outraged members of the park community flooded and demanded more land after the proposed structure had been rejected.\n\nCouncil spokesman Mike Diaz said the council had therefore agreed to"
# ' The existing seven-story building has been "tree-lined" on both sides, and new condos would include retail and restaurants.\n\nThe change would lower parking in a large downtown spike zone.'
# True
